# Merging centers

This notebook contains my tentative for merging close detected foci.

In [15]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
import numpy as np
import tensorflow as tf
import sys
from scipy.spatial import cKDTree
import scipy.spatial

my_path = Path.cwd().parent.parent / "src"
sys.path.append(str(my_path))

from visualization import visualization 
from utils import utils
from detection import blob_detection
from pairing import pairing

%matplotlib inline

In [ ]:
ROOT = Path(r"K:\users\voland\data\current\TgCetnEos_CenSpark_H2B_4h-24h-48hpf\20260506CFHa_CetnEos_H2BmCherry_CS_24hpf\3e1\subsets")

VOL_PATH = ROOT/"s1_20260506CFHa_CetnEos_H2BmCherry_CS_24hpf.lif - 3e1-1.tif"

In [3]:
vol = tifffile.imread(VOL_PATH)
scale = utils.get_pixel_size(VOL_PATH)
vol.shape

(69, 4, 438, 571)

In [6]:
blob_centers1 = blob_detection.frame_blob_detection(vol[0],"dog",0.1)
blob_centers1

array([[  0.        , 102.        , 265.        ,   1.73205081],
       [  0.        , 252.        , 427.        ,   1.73205081],
       [  0.        ,  95.        ,  51.        ,   1.73205081],
       [  0.        ,   9.        , 173.        ,   1.73205081],
       [  0.        , 290.        , 406.        ,   1.73205081],
       [  2.        ,  92.        , 294.        ,   1.73205081],
       [  0.        , 106.        , 315.        ,   1.73205081],
       [  2.        , 342.        , 383.        ,   1.73205081],
       [  1.        , 311.        , 395.        ,   1.73205081],
       [  0.        , 376.        , 371.        ,   1.73205081],
       [  0.        , 196.        , 303.        ,   1.73205081],
       [  1.        , 403.        , 388.        ,   1.73205081],
       [  1.        , 323.        , 372.        ,   1.73205081],
       [  1.        , 400.        , 385.        ,   1.73205081],
       [  1.        , 325.        , 385.        ,   1.73205081],
       [  1.        , 325

Let's get the index of the spots that are closer than a set merging_distance. These spots should be merged as they are too close from each others.

In [ ]:
merged = True
while merged:
    merged = False
    tree = cKDTree(centers_um)
    
    # Find all pairs within merging_distance
    pairs = tree.query_pairs(r=2, output_type='ndarray')  # shape (M, 2)
    
    if len(pairs) == 0:
        break
    
    unmerged = np.zeros(len(centers_um), dtype=bool)
    merge_map = {}  # idx -> merged result

    for i, j in pairs:
        if unmerged[i] or unmerged[j]:
            continue
        # Merge i and j
        merge_map[i] = (blob_centers1[i] + blob_centers1[j]) / 2
        unmerged[i] = True
        unmerged[j] = True
        merged = True

    # Keep unmerged spots + new merged spots
    unmerged_centers = blob_centers1[~unmerged]
    new_merged = np.array(list(merge_map.values()))
    blob_centers1 = np.concatenate([unmerged_centers, new_merged]) if len(new_merged) > 0 else unmerged_centers
    centers_um = blob_centers1[:, :3] * scale

In [8]:
blob_centers2 = blob_detection.frame_blob_detection(vol[1],"dog",0.1)

In [133]:
points1, points2 = blob_centers1, blob_centers2

In [55]:
point1 = points1[2]

In [54]:
dist_matrix = scipy.spatial.distance_matrix(points1, points2)
dist_matrix

array([[  1.41421356, 109.55363983, 215.11392331, ..., 297.60880363,
        263.88444441,  57.41950888],
       [219.36727194, 177.79201332, 408.38462263, ...,  99.01010049,
         65.62773804, 229.41011312],
       [215.14878573, 233.8054747 ,   1.        , ..., 455.7038512 ,
        428.92190431, 254.56629785],
       ...,
       [329.62099448, 238.80117253, 460.56704181, ...,  72.42237223,
         98.35141077, 360.68129976],
       [281.68599539, 203.88967605, 435.93004026, ...,  23.19482701,
         33.61547263, 306.47348988],
       [144.13188405,  46.10856753, 278.49236973, ..., 178.21616088,
        151.64761785, 187.96542235]], shape=(202, 198))

In [56]:
z_distnace = np.abs(point1[0] -points2[:,0])
candidate_spot = np.argwhere(z_distnace < 2)
candidate_spot

array([[  0],
       [  1],
       [  2],
       [  3],
       [  4],
       [  5],
       [  6],
       [  8],
       [  9],
       [ 10],
       [ 11],
       [ 12],
       [ 13],
       [ 14],
       [ 15],
       [ 16],
       [ 17],
       [ 18],
       [ 19],
       [ 20],
       [ 21],
       [ 22],
       [ 23],
       [ 24],
       [ 26],
       [ 27],
       [ 29],
       [ 31],
       [ 32],
       [ 33],
       [ 34],
       [ 35],
       [ 36],
       [ 37],
       [ 38],
       [ 39],
       [ 40],
       [ 41],
       [ 42],
       [ 43],
       [ 44],
       [ 45],
       [ 46],
       [ 47],
       [ 48],
       [ 49],
       [ 50],
       [ 51],
       [ 52],
       [ 53],
       [ 54],
       [ 55],
       [ 56],
       [ 57],
       [ 58],
       [ 59],
       [ 60],
       [ 61],
       [ 62],
       [ 63],
       [ 64],
       [ 65],
       [ 66],
       [ 67],
       [ 68],
       [ 69],
       [ 71],
       [ 72],
       [ 73],
       [ 74],
       [ 75],
      

In [59]:
candidate_spot[dist_matrix[1, candidate_spot].argmin()]

array([91])

In [70]:
opti_row, opti_col = scipy.optimize.linear_sum_assignment(dist_matrix)
opti_col

array([  0,  91,   2,  23,  10,   7,   4,  25, 197,  60, 136, 146,  34,
        26,  38,  44,  54,  24, 174,  31,  21, 112,  94,  73,  90,  79,
       159,  50, 173, 131, 168,  27, 169,  56, 189,  30,  22,  82,  29,
        48,  92,  49, 107,  46,  14,  78,  28, 144, 115, 149,  93,  32,
        68, 102, 145, 172, 120,  19, 137, 129, 105,  37,  67, 166, 170,
        95,  74,   9,  69,  33,  66, 148,  96, 196,  87,   6, 187,  40,
       161,  98, 184,  81,  86,  45, 104, 130, 179,  97,  36, 176,  53,
       178, 109, 116, 181, 194, 110,  13,  18,  47, 152,  62, 114,  88,
       190,  72, 103, 123, 185,  89, 122,  35, 138, 163,  65,  15, 132,
       126, 182, 188,  51,  20,   1, 150,  84,  41, 119,  99,  64, 133,
       191, 175,  75,  11, 141, 121,   3,  76, 156, 100,  71,  59,  58,
        63, 167,  52, 154,  43, 158, 177, 195, 101, 142, 165, 151, 171,
        39,   8,  61, 164, 193, 143,  17,  55, 192,  80,  57, 140, 139,
       106, 155,  16,  85,  12, 118, 111, 117, 180, 162, 153,  4

In [134]:
point1 = points1[2]
dist_matrix = scipy.spatial.distance_matrix(points1[:,1:3], points2[:,1:3])
opti_row = []
opti_col = []
for i, point in enumerate(points1):
    z_distnace = np.abs(point[0] - points2[:,0])
    candidate_spot = np.argwhere(z_distnace < 2)
    nneibourgh = candidate_spot[dist_matrix[i, candidate_spot].argmin()]
    if dist_matrix[i,nneibourgh] < 40:
        opti_row.append(i)
        opti_col.append(int(nneibourgh))

C:\Users\voland\AppData\Local\Temp\3\ipykernel_2120\1816259762.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  opti_col.append(int(nneibourgh))


In [135]:
np.array(opti_row)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [136]:
np.array(opti_col)

array([  0,  91,   2,  80,  10,   7,   4,  28,  31,  60,  45, 146, 126,
        26, 125, 139, 134,  22,  42, 174, 102,  21, 124,  94,  17,  40,
        79, 136, 134, 145, 131,  78,  27, 169,  75, 189, 102,  59,  74,
       185, 101,  92, 115, 133,  11,  14,  33, 181, 144,  69,  78,  50,
        29,   9,  59, 145, 103, 137,  22,  98, 122, 105, 134,  83,  67,
       166,  15,  95,  32,   9,  69, 139,  96, 103, 111, 196, 128,  59,
       153,  12,  55, 111,  93,  42, 149,  53, 104, 186,  38,  31,  72,
       110,  53, 178,   8,  49,  27, 196, 119,  41, 137,  47, 184,  62,
       169,  62, 101, 149, 130, 173, 185,  76, 170,  86, 105,  87,   6,
       104, 148,  76, 182,  79, 188,  51, 156, 137,  62, 100, 119, 111,
       141, 152, 158,   1, 166, 175,  75,  40, 107, 149,  80, 125, 156,
        90, 182, 131, 184, 169, 167, 119, 170, 164, 158,  58, 195, 137,
       142, 168,  24,  69,  39,   5,  92, 164,  18,  33,  17,  98, 115,
        80, 117,  21, 106, 112, 155, 151,  90, 154, 118, 126,  4

In [130]:
ans = {num: [i for i, x in enumerate(opti_col) if x == num] for num in set(opti_col) if opti_col.count(num) > 1}  
ans

{5: [161, 194],
 9: [53, 69],
 17: [24, 166, 195],
 18: [164, 196],
 21: [21, 171],
 22: [17, 58],
 27: [32, 96],
 31: [8, 89],
 33: [46, 165],
 40: [25, 137],
 41: [99, 200],
 42: [18, 83, 184],
 53: [85, 92],
 59: [37, 54, 77],
 62: [103, 105, 126],
 69: [49, 70, 159],
 75: [34, 136],
 76: [111, 119],
 78: [31, 50],
 79: [26, 121],
 80: [3, 140, 169],
 83: [63, 190],
 90: [143, 176],
 92: [41, 162],
 95: [67, 181],
 98: [59, 167],
 101: [40, 106],
 102: [20, 36],
 103: [56, 73],
 104: [86, 117],
 105: [61, 114],
 106: [172, 199],
 111: [74, 81, 129],
 115: [42, 168],
 119: [98, 128, 149],
 122: [60, 192],
 124: [22, 187],
 125: [14, 141],
 126: [12, 179],
 131: [30, 145],
 133: [43, 197],
 134: [16, 28, 62, 185],
 137: [57, 100, 125, 155],
 139: [15, 71, 183],
 145: [29, 55],
 149: [84, 107, 139],
 156: [124, 142],
 158: [132, 152],
 164: [151, 163],
 166: [65, 134],
 169: [33, 104, 147],
 170: [112, 150],
 174: [19, 198],
 182: [120, 144],
 184: [102, 146],
 185: [39, 110],
 189: [3

In [93]:
7 in ans.keys()

False

In [131]:
new_opti_row = []
new_opti_col = []
for i in range(len(opti_row)):
    row = opti_row[i]
    col = opti_col[i]
    if col not in new_opti_col:
        if col in ans.keys():
            closest_row = dist_matrix[:,col].argmin()
            new_opti_row.append(closest_row)
        else:
            new_opti_row.append(row)
            new_opti_col.append(col)

In [132]:
len(new_opti_row)

201

In [108]:
dist_matrix[3,80]

np.float64(37.013511046643494)

In [109]:
dist_matrix[3,23]

np.float64(100.12492197250393)

In [114]:
max_pairing_distance_xy = 40
max_pairing_distance_z = 2

In [123]:
from scipy.spatial.distance import cdist

n1 = len(points1)
n2 = len(points2)

xy_dist = cdist(points1[:, 1:3], points2[:, 1:3])

# nearest valid neighbor of points1 in points2
nn12 = np.full(n1, -1, dtype=int)

for i in range(n1):

    valid = (
        np.abs(points1[i, 0] - points2[:, 0])
        <= max_pairing_distance_z
    )

    if not np.any(valid):
        continue

    candidates = np.where(valid)[0]

    best = candidates[
        np.argmin(xy_dist[i, candidates])
    ]

    if xy_dist[i, best] <= max_pairing_distance_xy:
        nn12[i] = best

# nearest valid neighbor of points2 in points1
nn21 = np.full(n2, -1, dtype=int)

for j in range(n2):

    valid = (
        np.abs(points2[j, 0] - points1[:, 0])
        <= max_pairing_distance_z
    )

    if not np.any(valid):
        continue

    candidates = np.where(valid)[0]

    best = candidates[
        np.argmin(xy_dist[candidates, j])
    ]

    if xy_dist[best, j] <= max_pairing_distance_xy:
        nn21[j] = best

matched1 = []
matched2 = []

for i, j in enumerate(nn12):

    if j == -1:
        continue

    if nn21[j] == i:
        matched1.append(i)
        matched2.append(j)

matched1 = np.asarray(matched1)
matched2 = np.asarray(matched2)

distances_xy = np.linalg.norm(
    points1[matched1, 1:3]
    - points2[matched2, 1:3],
    axis=1,
)

distances_z = np.abs(
    points1[matched1, 0]
    - points2[matched2, 0]
)

In [126]:
len(matched1)

114

In [127]:
matched_2

array([  0,  91,   2,  10,   7,   4,  28,  60,  45, 146, 126,  26, 125,
       139,  22, 102,  21,  94,  17,  79, 136, 131,  78,  27,  59,  74,
       101,  92, 115, 133,  11,  14,  33,  93,  29, 145,  98, 105,  83,
        67,  15,  95,  32,   9,  69,  96, 103, 196, 128, 153,  12, 186,
        38,  31,  72, 110,  53, 178,   8,  49,  41,  47, 184, 130, 173,
       185,   6, 104, 148, 182, 188,  51, 156,  62, 100, 119, 111, 141,
       152,   1, 166, 175,  40, 107, 149, 169, 167, 164, 177, 195, 137,
       142,  24,  39,   5,  18,  80, 117, 112, 155, 151,  90, 154, 118,
        46, 189,  42, 134, 108, 124, 138, 135, 122,  70, 174, 106])

In [173]:
matched1 = []
matched2 = []
for i, point1 in enumerate(points1):
    # Get the nearest neighour of my point 1
    z_distance12 = np.abs(point1[0] - points2[:,0])
    candidates2 = np.argwhere(z_distance12 < max_pairing_distance_z)
    closest2 = int(candidates2[dist_matrix[i, candidates2].argmin()][0])

    # Check who is the nearest neighour of my point 2
    z_distance21 = np.abs(points2[closest2,0] - points1[:,0])
    candidates1 = np.argwhere(z_distance21 < max_pairing_distance_z)
    closest1 = candidates1[dist_matrix[candidates1,closest2].argmin()]

    # If both points are the nearest neighour of the other then pair them 
    if i == closest1:
        matched1.append(i)
        matched2.append(closest2)

In [174]:
matched2

[0,
 91,
 2,
 10,
 7,
 4,
 28,
 60,
 45,
 146,
 126,
 26,
 125,
 139,
 22,
 102,
 21,
 94,
 17,
 79,
 136,
 131,
 78,
 27,
 59,
 74,
 101,
 92,
 115,
 133,
 11,
 14,
 33,
 50,
 29,
 9,
 145,
 98,
 105,
 83,
 67,
 15,
 95,
 32,
 69,
 96,
 103,
 196,
 128,
 153,
 12,
 104,
 186,
 38,
 31,
 72,
 110,
 53,
 8,
 49,
 41,
 47,
 184,
 130,
 173,
 185,
 6,
 148,
 182,
 188,
 51,
 156,
 62,
 100,
 119,
 111,
 141,
 152,
 1,
 166,
 175,
 40,
 107,
 149,
 169,
 167,
 164,
 195,
 137,
 142,
 24,
 39,
 5,
 18,
 80,
 117,
 112,
 155,
 151,
 90,
 154,
 118,
 46,
 189,
 42,
 134,
 108,
 124,
 138,
 135,
 122,
 70,
 174,
 106]